# Regressão Logística - Classificação Binária

Este notebook utiliza o dataset **Breast Cancer Wisconsin** (disponível no sklearn) para classificar tumores como **malignos** ou **benignos** usando Regressão Logística. O processo inclui padronização das features, divisão treino-teste, avaliação por matriz de confusão e análise da Curva ROC.

## 1. Imports

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix,
                             ConfusionMatrixDisplay,
                             classification_report,
                             roc_curve, roc_auc_score)


## 2. Carregamento e Treinamento

In [ ]:
# Carrega o dataset de câncer de mama
dados = load_breast_cancer()

# x: matriz com as 30 features dos tumores
x = dados.data

# y: vetor alvo (0 = maligno, 1 = benigno)
y = dados.target

# Divide os dados em treino (70%) e teste (30%) com seed fixa para reprodutibilidade
x_treino, x_teste, y_treino, y_teste = train_test_split(
    x,
    y,
    test_size=0.3,
    random_state=42
)

# Cria o padronizador e ajusta com os dados de treino, depois transforma ambos
scaler = StandardScaler()
x_treino_escalonado = scaler.fit_transform(x_treino)
x_teste_escalonado = scaler.transform(x_teste)

# Instancia e treina o modelo de regressão logística
model = LogisticRegression()
model.fit(x_treino_escalonado, y_treino)

# Faz previsões e obtém probabilidades para cada classe
prev = model.predict(x_teste_escalonado)
prob = model.predict_proba(x_teste_escalonado)

# Exibe a classe prevista e as probabilidades para o primeiro paciente do teste
print(f'''Classe prevista para o primeiro paciente: {prev[0]}
Probabilidade [maligno, benigno]: {prob[0]}''')

## 3. Matriz de Confusão

In [ ]:
# Calcula a matriz de confusão comparando valores reais e previstos
matriz = confusion_matrix(y_teste, prev)

# Cria o objeto para exibir a matriz com os nomes das classes
grafic = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=dados.target_names
)

# Plota a matriz de confusão com mapa de cores azul
grafic.plot(cmap='Blues')
plt.title('Matriz de Confusão')
plt.show()

# Exibe o relatório completo com precisão, recall, f1-score e suporte
print(classification_report(y_teste, prev, target_names=dados.target_names))

## 4. Curva ROC e AUC

In [ ]:
# Extrai as probabilidades da classe positiva (benigno = 1)
prob_positivas = prob[:, 1]

# Calcula os pontos da curva: Taxa de Falsos Positivos e Taxa de Verdadeiros Positivos
tfp, tvp, _ = roc_curve(y_teste, prob_positivas)

# Calcula a área sob a curva (AUC) — quanto maior, melhor o modelo
auc_score = roc_auc_score(y_teste, prob_positivas)

# Plota a curva ROC com o valor da AUC no rótulo
plt.figure(figsize=(8, 6))
plt.plot(tfp, tvp, color='darkorange',
         lw=2,
          label=f'Curva ROC (AUC = {auc_score:.3f})')

# Linha diagonal pontilhada: representa um classificador aleatório (baseline)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('Taxa de Falsos Positivos (TFP)')
plt.ylabel('Taxa de Verdadeiros Positivos (TVP)')
plt.title('Curva ROC - Diagnóstico de câncer')
plt.legend(loc='lower right')
plt.grid()
plt.show()